<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/orientation_tuning_three_modalities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Orientation tuning across three recording scales

A second **confidence-building** notebook for the [OpenScope Community Predictive
Processing](https://allenneuraldynamics.github.io/openscope-community-predictive-processing/)
dataset (companion to the receptive-field notebook). Orientation selectivity is
the canonical property of mouse visual cortex, so it is an ideal known-answer
check: real V1/LM neurons should be **orientation-tuned**, and that tuning should
**reproduce across independent trial halves**.

We compute direction/orientation tuning in all three modalities from the
drifting-grating sweep, and quantify it with OSI, DSI, and split-half tuning
reliability:

| Modality | DANDI | Signal |
|---|---|---|
| Neuropixels ecephys | [001637](https://dandiarchive.org/dandiset/001637) | spike rate |
| Mesoscope 2-photon | [001768](https://dandiarchive.org/dandiset/001768) | ΔF/F (jGCaMP8s, soma) |
| SLAP2 | [001424](https://dandiarchive.org/dandiset/001424) | ΔF/F (iGluSnFR, glutamate) |

**The stimulus.** Every session's `Control block 1` (`standard_control`) contains a
full **14-direction drifting-grating sweep** (0–315° in 22.5° steps, TF = 2 Hz,
~68 trials/direction) plus blanks — a proper orientation/direction-tuning battery.
In SLAP2 the same directions live in the full-field (large-diameter) gratings of
the monolithic `gratings` stream.

**Runtime:** CPU is fine; data streams over HTTP. ~8–12 min end to end.

In [ ]:
#@title Install dependencies
!pip -q install pynwb h5py remfile requests pandas numpy matplotlib

In [ ]:
#@title Streaming helpers + tuning metrics
import h5py, remfile, requests, numpy as np, pandas as pd

def s3_url(dandiset, asset_id, version="draft"):
    r = requests.get(f"https://api.dandiarchive.org/api/dandisets/{dandiset}/versions/{version}/assets/{asset_id}/download/",
                     allow_redirects=False, timeout=30)
    return r.headers["Location"]
def open_nwb(dandiset, asset_id):
    return h5py.File(remfile.File(s3_url(dandiset, asset_id)), "r")
def col(group, name):
    v = group[name][:]
    return np.array([x.decode() if isinstance(x, bytes) else x for x in v])

def tuning_metrics(curve, theta):
    """OSI/DSI (vector method) + preferred orientation/direction from a tuning curve."""
    r = np.clip(curve, 0, None)
    if r.sum() == 0: return dict(osi=0, dsi=0, pref_ori=np.nan, pref_dir=np.nan)
    osi = np.abs(np.sum(r*np.exp(2j*theta)))/r.sum()      # 2*theta -> orientation (180-periodic)
    dsi = np.abs(np.sum(r*np.exp(1j*theta)))/r.sum()      # theta   -> direction  (360-periodic)
    return dict(osi=osi, dsi=dsi,
                pref_ori=np.degrees(np.angle(np.sum(r*np.exp(2j*theta))))/2 % 180,
                pref_dir=np.degrees(np.angle(np.sum(r*np.exp(1j*theta)))) % 360)

def split_half_tuning(ev, ori_labels, oris):
    """Correlation between tuning curves from even- and odd-numbered trials."""
    a = np.array([np.nanmean(ev[np.where(ori_labels==o)[0][0::2]]) for o in oris])
    b = np.array([np.nanmean(ev[np.where(ori_labels==o)[0][1::2]]) for o in oris])
    if np.nanstd(a) < 1e-9 or np.nanstd(b) < 1e-9: return np.nan
    return np.corrcoef(a, b)[0, 1]

print("helpers ready")

## 1 · Neuropixels ecephys

Spike-rate tuning from the drifting-grating sweep. Response = mean rate over the
grating (50 ms–offset), minus a pre-stimulus baseline. Visual units are assigned
to a CCF area via the corrected per-probe electrode mapping.

In [ ]:
#@title Ecephys tuning — session sub-830851
import re
ECE = "9b9e8abe-7b43-47f1-b8e1-4114f87898a1"
fh = open_nwb("001637", ECE)
g = fh["intervals"]["Control block 1_presentations"]
O = g["Orientation"][:].astype(float); C = g["contrast"][:].astype(float); t0 = g["start_time"][:]
grat = C > 0
oris = np.unique(O[grat]); dur = float(np.median(g["stop_time"][:]-g["start_time"][:]))
tg = t0[grat]; Og = O[grat]

U = fh["units"]; st = U["spike_times"][:]; sti = U["spike_times_index"][:]
def spikes(i): return st[(0 if i==0 else sti[i-1]):sti[i]]
n = len(sti); qc = U["default_qc"][:] if "default_qc" in U else np.ones(n, bool)
E = fh["general/extracellular_ephys/electrodes"]; eloc = col(E,"location"); egroup = col(E,"group_name")
dev = col(U,"device_name"); eci = U["extremum_channel_index"][:]
groups = sorted(set(egroup), key=lambda gn: np.where(egroup==gn)[0][0])
off = {gn: np.where(egroup==gn)[0][0] for gn in groups}; bl = {gn: int((egroup==gn).sum()) for gn in groups}
def g4(d):
    for gn in groups:
        if d[-1].lower() in gn.lower(): return gn
    return groups[0]
d2g = {d: g4(d) for d in set(dev)}
u_region = np.array([eloc[off[d2g[dev[i]]]+min(int(eci[i]), bl[d2g[dev[i]]]-1)] for i in range(n)])
sel = np.where(qc & np.array([bool(re.match("VIS", r)) for r in u_region]))[0]

RESP, BASE = (0.05, dur), (-0.25, -0.02)
tc_e = np.zeros((len(sel), len(oris))); rows = []
for ui, u in enumerate(sel):
    sp = spikes(u)
    ev = ((np.searchsorted(sp, tg+RESP[1])-np.searchsorted(sp, tg+RESP[0]))/(RESP[1]-RESP[0])
          - (np.searchsorted(sp, tg+BASE[1])-np.searchsorted(sp, tg+BASE[0]))/(BASE[1]-BASE[0]))
    tc_e[ui] = [np.mean(ev[Og==o]) for o in oris]
    m = tuning_metrics(tc_e[ui], oris); m.update(unit=u, region=u_region[u],
        peak=tc_e[ui].max(), splithalf=split_half_tuning(ev, Og, oris))
    rows.append(m)
Te = pd.DataFrame(rows)
print(f"{len(Te)} visual units | median OSI={Te.osi.median():.2f} | median split-half r={Te.splithalf.median():.2f}")
fh.close()

## 2 · Mesoscope 2-photon (somatic ΔF/F)

Same block, same 14 directions. Soma-masked ROIs; slower response window for
calcium.

In [ ]:
#@title Mesoscope tuning — session sub-832700, plane VISl_4
MESO = "83e0c8f3-5208-417c-87c4-bc4617b0f834"
fhm = open_nwb("001768", MESO)
gm = fhm["intervals"]["Control block 1_presentations"]
Om = gm["Orientation"][:].astype(float); Cm = gm["contrast"][:].astype(float); t0m = gm["start_time"][:]
gr = Cm > 0; oris_m = np.unique(Om[gr]); tgm = t0m[gr]; Ogm = Om[gr]
PLANE = "VISl_4"; p0 = fhm["processing"][PLANE]
dff = p0["dff_timeseries"]["dff_timeseries"]; data = dff["data"][:]; tsv = dff["timestamps"][:]
soma = np.where(p0["image_segmentation"]["roi_table"]["is_soma"][:].astype(bool))[0]
RESP_M, BASE_M = (0.1, 0.9), (-0.3, 0.0)
def resp(roi, times):
    tr = data[:, roi]; out = np.full(len(times), np.nan)
    for k, s in enumerate(times):
        rm = (tsv>=s+RESP_M[0])&(tsv<s+RESP_M[1]); bm = (tsv>=s+BASE_M[0])&(tsv<s+BASE_M[1])
        if rm.sum() and bm.sum(): out[k] = np.nanmean(tr[rm]) - np.nanmean(tr[bm])
    return out
tc_m = np.zeros((len(soma), len(oris_m))); rows = []
for ui, roi in enumerate(soma):
    ev = resp(roi, tgm); tc_m[ui] = [np.nanmean(ev[Ogm==o]) for o in oris_m]
    m = tuning_metrics(tc_m[ui], oris_m); m.update(roi=roi, peak=np.nanmax(tc_m[ui]),
        splithalf=split_half_tuning(ev, Ogm, oris_m)); rows.append(m)
Tm = pd.DataFrame(rows)
print(f"{len(Tm)} soma ROIs | median OSI={Tm.osi.median():.2f} | median split-half r={Tm.splithalf.median():.2f}")
fhm.close()

## 3 · SLAP2 (dendritic glutamate ΔF/F)

The 14 directions live in the full-field (diameter ≥ 30°) gratings of the
monolithic `gratings` stream. We use the best-yield session (sub-796630,
2025-10-01, DMD1) and the DMD1 timebase rebuild (see the RF notebook for why).
Note: orientation tuning is *more* reliable in SLAP2 than RF mapping was — many
trials per direction and a sustained response suit dendritic glutamate imaging.

In [ ]:
#@title SLAP2 tuning — session sub-796630 2025-10-01, DMD1
SLAP = "44871646-ca8d-440d-b970-5756ed7cb47e"
fhs = open_nwb("001424", SLAP)
gs = fhs["intervals/gratings"]
dia = gs["diameter"][:]; ori = gs["orientation"][:].astype(float); con = gs["contrast"][:].astype(float)
t0s = gs["start_time"][:]
big = (dia >= 30) & (con > 0)                       # full-field gratings = tuning battery
oris_s = np.unique(np.round(ori[big], 3)); Ogs = np.round(ori[big], 3)
DMD = "DMD1"; OFFSET = 0.115
dff1 = fhs[f"processing/ophys/Fluorescence_{DMD}/{DMD}_dFF"]; d = dff1["data"][:]
ts = dff1["timestamps"][:]; ts_o = fhs["processing/ophys/Fluorescence_DMD2/DMD2_dFF"]["timestamps"][:]
if ts[-1] < 0.6*ts_o[-1]: ts = np.linspace(ts_o[0], ts_o[-1], d.shape[0])   # rebuild compressed timebase
tgs = t0s[big] + OFFSET
RESP_S, BASE_S = (0.05, 0.35), (-0.25, -0.02)
def resp_s(roi, times):
    tr = d[:, roi]; out = np.full(len(times), np.nan)
    for k, s in enumerate(times):
        rm = (ts>=s+RESP_S[0])&(ts<s+RESP_S[1]); bm = (ts>=s+BASE_S[0])&(ts<s+BASE_S[1])
        if rm.sum() and bm.sum():
            rv, bv = tr[rm], tr[bm]
            if np.isfinite(rv).any() and np.isfinite(bv).any(): out[k] = np.nanmean(rv) - np.nanmean(bv)
    return out
tc_s = np.zeros((d.shape[1], len(oris_s))); rows = []
for ui in range(d.shape[1]):
    ev = resp_s(ui, tgs); tc_s[ui] = [np.nanmean(ev[Ogs==o]) for o in oris_s]
    m = tuning_metrics(tc_s[ui], oris_s); m.update(roi=ui, peak=np.nanmax(tc_s[ui]),
        splithalf=split_half_tuning(ev, Ogs, oris_s)); rows.append(m)
Ts = pd.DataFrame(rows)
print(f"{len(Ts)} ROIs | median OSI={Ts.osi.median():.2f} | median split-half r={Ts.splithalf.median():.2f}")
fhs.close()

## 4 · The figure — tuning across all three scales

Top three rows: the most reliable example tuning curves per modality (polar,
rectified). Bottom row: population distributions of OSI, split-half reliability,
and preferred orientation.

In [ ]:
#@title Plot
import matplotlib.pyplot as plt
specs = [("Neuropixels\nspikes",      Te, tc_e, "unit", oris,   "#08519c", 0.5, 3,   0.15),
         ("Mesoscope 2p\nΔF/F (soma)", Tm, tc_m, "roi",  oris_m, "#238b45", 0.5, 0.1, 0.15),
         ("SLAP2\nglutamate ΔF/F",     Ts, tc_s, "roi",  oris_s, "#d94801", 0.4, 0.08,0.2)]
fig = plt.figure(figsize=(13, 9.0))
for ri,(name,Tdf,tcm,idc,ors,color,shthr,pk,osi) in enumerate(specs):
    well = Tdf[(Tdf.splithalf>shthr)&(Tdf.peak>pk)&(Tdf.osi>osi)].sort_values("splithalf",ascending=False).head(3)
    for j,(_,row) in enumerate(well.iterrows()):
        ax = fig.add_subplot(4,6,ri*6+j*2+1, projection="polar")
        ui = int(np.where(Tdf[idc].values==row[idc])[0][0])
        c = np.clip(tcm[ui],0,None); thc=np.append(ors,ors[0]); cc=np.append(c,c[0])
        ax.plot(thc,cc,color=color,lw=1.6); ax.fill(thc,cc,color=color,alpha=0.2)
        ax.set_theta_zero_location("E"); ax.set_xticks(np.linspace(0,2*np.pi,8,endpoint=False))
        ax.set_xticklabels([]); ax.set_yticklabels([]); ax.tick_params(labelsize=5)
        ax.set_title(f"OSI={row.osi:.2f} · r={row.splithalf:.2f}", fontsize=6.5, pad=4)
    fig.text(0.012, 0.85-ri*0.205, name, rotation=90, va="center", ha="center",
             fontsize=9, fontweight="bold", color=color)
axA=fig.add_subplot(4,3,10); axB=fig.add_subplot(4,3,11); axC=fig.add_subplot(4,3,12)
for name,Tdf,_,_,_,color,shthr,pk,osi in specs:
    lab=name.split("\n")[0]
    axA.hist(Tdf.osi.dropna(),bins=np.linspace(0,1,20),histtype="step",lw=2,color=color,density=True,label=f"{lab} ({Tdf.osi.median():.2f})")
    axB.hist(Tdf.splithalf.dropna(),bins=np.linspace(-0.5,1,25),histtype="step",lw=2,color=color,density=True,label=f"{lab} ({Tdf.splithalf.median():.2f})")
    w=Tdf[(Tdf.splithalf>shthr)&(Tdf.peak>pk)&(Tdf.osi>osi)]
    axC.hist(w.pref_ori.dropna(),bins=np.arange(0,181,20),histtype="step",lw=2,color=color,density=True)
axA.set_xlabel("OSI"); axA.set_ylabel("density"); axA.set_title("A · Orientation selectivity",fontsize=9,loc="left"); axA.legend(fontsize=6.2)
axB.axvline(0,color="gray",lw=0.6,ls=":"); axB.set_xlabel("split-half tuning r"); axB.set_title("B · Tuning reproducibility",fontsize=9,loc="left"); axB.legend(fontsize=6.2)
axC.set_xlabel("preferred orientation (°)"); axC.set_title("C · Preferred orientations",fontsize=9,loc="left"); axC.set_xticks([0,45,90,135,180])
fig.suptitle("Orientation tuning across three recording scales — drifting gratings, full direction sweep",fontsize=11,y=0.985)
fig.subplots_adjust(left=0.06,right=0.98,top=0.93,bottom=0.065,wspace=0.42,hspace=0.7)
fig.savefig("orientation_tuning_three_modalities.png",dpi=200,bbox_inches="tight")
plt.show()

## Takeaway

All three modalities show **strong, highly reproducible orientation tuning** —
clean bilobed polar curves, OSI well above zero, split-half reliability far from
the r = 0 null, and preferred orientations spanning 0–180°.

| Modality | median OSI | median DSI | split-half r |
|---|---|---|---|
| Neuropixels (spikes) | 0.38 | 0.18 | 0.79 |
| Mesoscope (ΔF/F soma) | 0.63 | 0.40 | 0.60 |
| SLAP2 (glutamate) | 0.49 | 0.31 | 0.91 |

**Complementary to the RF check.** SLAP2 was the *weakest* arm for receptive-field
mapping (sparse, brief patch stimuli → low SNR per position) but is the *most
reliable* here (r = 0.91): orientation tuning uses many trials per condition and a
sustained drifting-grating response, which suits dendritic glutamate imaging.
Mesoscope shows the sharpest selectivity (somatic calcium integrates the sustained
response). Together, the RF check validates **spatial** sensitivity and the tuning
check validates **feature** sensitivity — the pipeline reads real visual signals at
all three scales.